[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/prompts/few_shot_prompting.ipynb)


<center>
<p style="text-align:center">
<img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
<br>
<br>
<a href="https://arize.com/docs/phoenix/">文档</a>
|
<a href="https://github.com/Arize-ai/phoenix">GitHub</a>
|
<a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
</p>
</center>
<h1 align="center">少样本提示教程</h1>

少样本提示是提示词工程中的一种有效方法：在提示词中提供若干示例，帮助 LLM 更好地完成任务。

零样本提示要求模型在没有示例的情况下理解任务，单样本提示只提供一个示例，而少样本提示通过多个示例引导模型生成更准确的回答。

本教程将：
- 在评论数据集上探索不同提示策略对情感分析效果的影响。
- 运行评测，衡量提示词如何影响模型表现。
- 使用 Phoenix 记录提示词与实验随时间的变化。

完成后，你将理解结构化提示如何显著改善应用效果。

⚠️ 本教程需要 OpenAI API 密钥。

开始吧！🚀

# 配置依赖与密钥

In [ ]:
!pip install -qqq "arize-phoenix>=8.0.0" datasets openinference-instrumentation-openai

接下来连接 Phoenix。下面的代码连接到 Phoenix Cloud 实例；你也可以选择[连接自托管 Phoenix 实例](https://arize.com/docs/phoenix/deployment)。

In [ ]:
import os
from getpass import getpass

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com"
if not os.environ.get("PHOENIX_CLIENT_HEADERS"):
    os.environ["PHOENIX_CLIENT_HEADERS"] = "api_key=" + getpass("请输入 Phoenix API 密钥：")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("请输入 OpenAI API 密钥：")

# 将数据集加载到 Phoenix
此数据集包含评论及其对应的情感标签。本 Notebook 会始终使用同一数据集评估不同提示技术的影响，并逐轮改进方案。

这里还导入 Phoenix 客户端，以便直接在 Notebook 中创建和修改提示词，并将修改同步到 Phoenix 界面。

In [ ]:
from datasets import load_dataset

ds = load_dataset("syeddula/fridgeReviews")["train"]
ds = ds.to_pandas()
ds.head()

# 配置 Phoenix 客户端

In [ ]:
import uuid

from phoenix.client import Client
from phoenix.client import Client as PhoenixClient

unique_id = uuid.uuid4()

# 将数据集上传到 Phoenix
dataset = Client().datasets.create_dataset(
    dataframe=ds,
    input_keys=["Review"],
    output_keys=["Sentiment"],
    name=f"review-classification-{unique_id}",
)

# 零样本提示


零样本提示要求语言模型在没有任何示例的情况下执行任务。模型仅依靠预训练知识生成回答，适合无需提供具体指导就要快速预测的场景。

本节将在情感分析数据集上使用零样本提示，不提供带标签的示例，直接要求模型把评论分为正面、负面或中性。随后评估模型仅凭提示词理解任务的能力。

In [ ]:
from openai import OpenAI
from openai.types.chat.completion_create_params import CompletionCreateParamsBase
from phoenix.client.types import PromptVersion

params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "你是评论情感评估器。判断评论为正面、负面还是中性，仅返回 positive、negative 或 neutral 这三个英文分类之一。",
        },
        {"role": "user", "content": "{{Review}}"},
    ],
)

prompt_identifier = "fridge-sentiment-reviews"

prompt = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="用于按情感对评论分类的提示词。",
    version=PromptVersion.from_openai(params),
)

此时，初始提示词已出现在 Phoenix 的提示词页面。后续的任何修改都会记录在**版本（Versions）**中，便于跟踪和比较变化。

Phoenix 存储的不只有提示词文本，还包括模板、模型配置和响应格式等关键信息，让输出生成过程保持结构化和一致性。


![提示词第 1 版](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/s-few-shot-6.png)

接下来为实验定义任务和评估器。

由于数据集含有真实标签，可以使用一个简单函数检查任务输出是否与预期输出一致。

In [ ]:
def zero_shot_prompt(input):
    client = OpenAI()
    resp = client.chat.completions.create(**prompt.format(variables={"Review": input["Review"]}))
    return resp.choices[0].message.content.strip()


def evaluate_response(output, expected):
    return output.lower() == expected["Sentiment"].lower()

如果希望为代码接入埋点，可以运行下面的单元。运行提示词和评测并不依赖这一步，但埋点可以提供追踪可视化，帮助深入理解模型行为。

In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from phoenix.otel import register

tracer_provider = register(project_name="few-shot-examples")
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

最后运行实验，并在 Phoenix 中查看结果。

In [ ]:
import nest_asyncio
from phoenix.client.experiments import run_experiment

nest_asyncio.apply()

initial_experiment = run_experiment(
    dataset,
    task=zero_shot_prompt,
    evaluators=[evaluate_response],
    experiment_description="零样本提示",
    experiment_name="zero-shot-prompt",
    experiment_metadata={"prompt": "prompt_id=" + prompt.id},
)

后续各节会逐步改进提示词，以提升模型表现和数据集上的评测结果。

![首次实验](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/s-few-shot-5.png)

# 单样本提示

单样本提示通过一个示例引导模型回答。在提示词中加入一个带标签的示例，可让模型更清楚地理解任务，与零样本提示相比，有望生成更准确的预测。

本节会向情感分析任务提供一条带标签的评论作为参考，再评估这少量指导如何影响模型正确分类情感的能力。

In [ ]:
ds = load_dataset("syeddula/fridgeReviews")["test"]
one_shot_example = ds.to_pandas().sample(1)

In [ ]:
one_shot_template = """
"你是评论情感评估器。判断评论为正面、负面还是中性，仅返回 positive、negative 或 neutral 这三个英文分类之一。"

下面是一条评论及其情感标签的示例：

{examples}
"""

params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {"role": "system", "content": one_shot_template.format(examples=one_shot_example)},
        {"role": "user", "content": "{{Review}}"},
    ],
)

one_shot_prompt = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="用于按情感对评论分类的单样本提示词。",
    version=PromptVersion.from_openai(params),
)

在 Phoenix 的提示词页面可以看到更新后的版本。提示词加入了从测试数据集中随机选取的一个示例，帮助模型分类。

![第二个版本](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/s-few-shot-3.png)

与上一步相同，定义任务并运行评估器。这次使用更新后的单样本提示词，观察评测结果的变化。

In [ ]:
def one_shot_prompt_template(input):
    client = OpenAI()
    resp = client.chat.completions.create(
        **one_shot_prompt.format(variables={"Review": input["Review"]})
    )
    return resp.choices[0].message.content.strip()

In [ ]:
one_shot_experiment = run_experiment(
    dataset,
    task=one_shot_prompt_template,
    evaluators=[evaluate_response],
    experiment_description="单样本提示",
    experiment_name="one-shot-prompt",
    experiment_metadata={"prompt": "prompt_id=" + one_shot_prompt.id},
)

本次运行的评测结果略有改善。下一节将尝试进一步提升效果。

**注意：**有时结果也可能下降，这不一定意味着做错了。模型选择、随机抽取的测试示例以及模型自身行为等因素都会影响结果。

![单样本提示结果](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/s-few-shot-2.png)

# 少样本提示

最后探索少样本提示：在提示词中加入多个带标签的示例，以提升模型表现。通过展示同一任务的多个实例，帮助模型理解预期输出，生成更准确且一致的回答。

本节会向情感分析任务提供多条带标签的评论作为参考，帮助模型识别规律、正确分类情感。随后比较少样本、零样本和单样本提示的准确率，观察增加示例的影响。

In [ ]:
ds = load_dataset("syeddula/fridgeReviews")["test"]
few_shot_examples = ds.to_pandas().sample(10)

In [ ]:
few_shot_template = """
"你是评论情感评估器。判断评论为正面、负面还是中性，仅返回 positive、negative 或 neutral 这三个英文分类之一。"

下面是评论及其情感标签的示例：

{examples}
"""
params = CompletionCreateParamsBase(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {"role": "system", "content": few_shot_template.format(examples=few_shot_examples)},
        {"role": "user", "content": "{{Review}}"},
    ],
)

few_shot_prompt = PhoenixClient().prompts.create(
    name=prompt_identifier,
    prompt_description="用于按情感对评论分类的少样本提示词。",
    version=PromptVersion.from_openai(params),
)

更新后的提示词也保存在 Phoenix 中，可以清楚地查看逐步形成的线性版本历史。

![少样本提示版本](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/s-few-shot-4.png)

与前面一样，运行任务和评测。

In [ ]:
def few_shot_prompt_template(input):
    client = OpenAI()
    resp = client.chat.completions.create(
        **few_shot_prompt.format(variables={"Review": input["Review"]})
    )
    return resp.choices[0].message.content.strip()

In [ ]:
few_shot_experiment = run_experiment(
    dataset,
    task=few_shot_prompt_template,
    evaluators=[evaluate_response],
    experiment_description="少样本提示",
    experiment_name="few-shot-prompt",
    experiment_metadata={"prompt": "prompt_id=" + few_shot_prompt.id},
)

# 最终结果

在最后这次运行中，评测结果的提升最明显。提示词中的多个示例为模型提供了更清晰的指导，改善了情感分类效果。

注意：效果仍会波动，有时也可能下降。这不一定意味着出错，因为模型选择、随机抽取的测试示例和模型自身行为都会影响结果。

![最终结果](https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/s-few-shot-1.png)

接下来可以查看更多 [Phoenix 示例](https://arize.com/docs/phoenix/notebooks)。如果你还没有为项目加星，也欢迎[在 GitHub 上支持我们](https://github.com/Arize-ai/phoenix)！⭐️
